# Prototyping Functions to Ingest Data

In [43]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from typing import List
from fredapi import Fred

## EIA Spot Prices

In [44]:
def fetch_eia_series_pet(series_ids=['RBRTE', 'RWTC'], # ID's of series we're pulling (Brent, WTI)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)


    URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide


def fetch_eia_series_gas(series_ids = ["RNGWHHD"], # ID's of series we're pulling (Henry Hub)
                         frequency= 'weekly', # Frequency of the spot prices, either daily, weekly, or monthly.
                         length = 5000): # Timespan that we're pulling, max 5000 weeks
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)
    

    URL_BASE = f"https://api.eia.gov/v2/natural-gas/pri/fut/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_series_gas()
    

series,RNGWHHD
period,
1997-01-10,3.79
1997-01-17,4.19
1997-01-24,2.98
1997-01-31,2.91
1997-02-07,2.53
...,...
2026-05-15,2.86
2026-05-22,3.11
2026-05-29,3.16


## FRED Series (DXY, VIX, 10Y - 2Y Spread)

In [45]:
def fetch_fred_series(series_ids=['T10Y2Y','VIXCLS','DTWEXBGS'], #ID's of the series we want to pull, here 10Y-2Y spread, VIX, and DXY
                      frequency = 'W-FRI'  # How frequent we want the samples to be. (day -> "D", weekly, on friday -> "W-FRI", monthly -> "M", yearly -> "Y"
                      ):
    
    load_dotenv()
    FRED_API_KEY = os.getenv('FRED_API_KEY')
    fred = Fred(api_key= FRED_API_KEY)
    series_dict = {}
    for id in series_ids: # Create a dict with key as ID and values as the corresponding series
        series_dict[id] = fred.get_series(series_id=id)

    df = pd.DataFrame(series_dict) # Turn dictionary into a DF
    df.index = pd.to_datetime(df.index) # Convert index dtype to datetime
    df.index.name = 'period' # Convert index name to 'period' to match the EIA information

    df = df.resample(rule=frequency).last() # Resample to keep only the days at the end of the week

    df = df.dropna() # Drop any NaNs

    return df
fetch_fred_series()

,T10Y2Y,VIXCLS,DTWEXBGS
period,,,
2006-01-06,0.02,11.00,100.0241
2006-01-13,0.02,11.23,99.9675
2006-01-20,0.00,14.56,99.9017
2006-01-27,0.01,11.97,99.6433
2006-02-03,-0.05,12.96,100.1180
...,...,...,...
2026-05-15,0.50,18.43,119.2825
2026-05-22,0.43,16.70,119.2868
2026-05-29,0.47,15.32,118.8783


In [46]:
def fetch_eia_stock(series_ids=['WCESTUS1'], # ID's of series we're pulling (Week-end US Crude Inventory)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)

    URL_BASE = f'https://api.eia.gov/v2/petroleum/stoc/wstk/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}'

    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

fetch_eia_stock().head()

series,WCESTUS1
period,
1982-08-20,338764
1982-08-27,336138
1982-09-24,335586
1982-10-01,334786
1982-10-08,335260


In [47]:

def get_merged_df(eia_spot_pet_df=fetch_eia_series_pet(), eia_spot_gas_df=fetch_eia_series_gas(), eia_stock_df=fetch_eia_stock(),fred_df=fetch_fred_series()):
    # Get and merge eia and fred data
    df = pd.merge(left=eia_spot_pet_df, right=fred_df,left_index = True, right_index = True, how= 'inner')
    df = pd.merge(left=df, right=eia_stock_df, left_index = True, right_index = True, how = 'inner')
    df = pd.merge(left=df, right= eia_spot_gas_df, left_index = True, right_index = True, how = 'inner' )
    df = df.copy().reset_index().sort_values('period')

    # Get and merge opec meeting dates, and the time since a meeting (these are big events for the market)
        
    opec_dates = pd.read_csv('../src/regime_detection/data/opec_meetings.csv', parse_dates=['date']).sort_values('date')
    
    df = pd.merge_asof(df, opec_dates.assign(last_opec=opec_dates['date']),
                    left_on='period', right_on='date', direction='backward') #merge_asof used for timeseries data where the dates don't match, finds the closest date in the rightmost df and returns that value for all rows matches, here we use backward to get the most recent date, forward would get the soonest.
    
    df['days_since_opec'] = (df['period'] - df['last_opec']).dt.days
    df = df.drop(columns=['date', 'last_opec'])
    
    return df.set_index('period')

In [48]:
def compute_returns(df=get_merged_df(), method= 'log',series_ids= ['RBRTE','RWTC', 'RNGWHHD']):
    df = df.copy()
    for id in series_ids:
        df[id + '_log_return'] = np.log(df[id]/ df[id].shift(1))
    return df.dropna()
    
    


In [49]:
def rolling_z_scores(df, 
                     series_ids=['RBRTE_log_return','RWTC_log_return', 'RNGWHHD_log_return'],
                     window=63,
                     min_periods=1):
    df = df.copy()
    for id in series_ids:
      avg = df[id].rolling(window=window, min_periods=min_periods).mean()
      dev = df[id].rolling(window=window, min_periods=min_periods).std()
      df[id + '_rol_z_score'] = (df[id] - avg) / dev
    return df


        


In [50]:
def series_diff(df, series_ids=['WCESTUS1']):
    df = df.copy()
    for id in series_ids:
        df[id + '_w_change'] = df[id].diff()
    return df

series_diff(compute_returns(get_merged_df()))

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return,WCESTUS1_w_change
period,,,,,,,,,,,,
2007-03-16,60.87,57.94,-0.03,16.79,97.0407,311926,6.86,1.0,0.008579,-0.049004,-0.064903,NaN
2007-03-23,61.09,58.26,0.02,12.95,96.5662,311080,6.91,8.0,0.003608,0.005508,0.007262,-846.0
2007-03-30,66.10,64.18,0.07,14.64,96.3183,315387,7.32,15.0,0.078821,0.096776,0.057641,4307.0
2007-04-06,68.55,64.82,0.01,13.23,96.2526,316219,7.55,22.0,0.036395,0.009923,0.030937,832.0
2007-04-13,68.20,62.58,0.00,12.20,95.7110,315225,7.83,29.0,-0.005119,-0.035168,0.036415,-994.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,166.0,0.042981,0.027198,0.042864,-7863.0
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,173.0,0.000724,0.002091,0.083801,-3327.0
2026-05-29,97.05,93.45,0.47,15.32,118.8783,433712,3.16,180.0,-0.130784,-0.119577,0.015949,-7974.0


In [51]:
def lagged_features(df, series_ids= ['RBRTE','RWTC'], lags=[1,4,12]):
    df = df.copy()
    for id in series_ids:
        for lag in lags:
            df[id + f'{lag}_w_lag'] = df[id].shift(lag)
    return df

In [52]:
def rolling_vol(df, series_ids=['RBRTE_log_return','RWTC_log_return','RNGWHHD_log_return'], periods=[4,12]):
    df = df.copy()
    for id in series_ids:
        id = id.split("_")[0]
        for period in periods:
            df[id + f'_{period}_rol_vol'] = df[id].rolling(window=period,min_periods=1).std()
    return df
rolling_vol(compute_returns(get_merged_df()))




,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec,RBRTE_log_return,RWTC_log_return,RNGWHHD_log_return,RBRTE_4_rol_vol,RBRTE_12_rol_vol,RWTC_4_rol_vol,RWTC_12_rol_vol,RNGWHHD_4_rol_vol,RNGWHHD_12_rol_vol
period,,,,,,,,,,,,,,,,,
2007-03-16,60.87,57.94,-0.03,16.79,97.0407,311926,6.86,1.0,0.008579,-0.049004,-0.064903,NaN,NaN,NaN,NaN,NaN,NaN
2007-03-23,61.09,58.26,0.02,12.95,96.5662,311080,6.91,8.0,0.003608,0.005508,0.007262,0.155563,0.155563,0.226274,0.226274,0.035355,0.035355
2007-03-30,66.10,64.18,0.07,14.64,96.3183,315387,7.32,15.0,0.078821,0.096776,0.057641,2.958079,2.958079,3.513934,3.513934,0.252389,0.252389
2007-04-06,68.55,64.82,0.01,13.23,96.2526,316219,7.55,22.0,0.036395,0.009923,0.030937,3.798442,3.798442,3.706571,3.706571,0.331763,0.331763
2007-04-13,68.20,62.58,0.00,12.20,95.7110,315225,7.83,29.0,-0.005119,-0.035168,0.036415,3.438008,3.754673,2.954229,3.260626,0.388962,0.415126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-15,110.53,105.10,0.50,18.43,119.2825,445013,2.86,166.0,0.042981,0.027198,0.042864,5.834601,15.751444,4.674830,12.096976,0.086410,0.163677
2026-05-22,110.61,105.32,0.43,16.70,119.2868,441686,3.11,173.0,0.000724,0.002091,0.083801,5.753511,11.049915,1.537040,8.231440,0.196363,0.172423
2026-05-29,97.05,93.45,0.47,15.32,118.8783,433712,3.16,180.0,-0.130784,-0.119577,0.015949,6.374132,8.952674,5.566497,5.687959,0.200562,0.186661


In [53]:
def differentials(df, series_ids=['RBRTE', 'RWTC']):
    df = df.copy()
    pairings = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1:]] 
    for pair in pairings:
        id_a , id_b = pair 
        df[f'{id_a}_{id_b}_diff'] = df[id_a] - df[id_b]
    return df



In [54]:
series_ids=['RBRTE', 'RWTC', 'BDO']
combinations = []
for id in series_ids:
    for neg_id in series_ids[::-1]:
        pair = set((id, neg_id))
        if pair not in combinations and len(pair) == 2:
            combinations.append(pair)

print(combinations)


combinations = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1::]] # (x,y), get series id and entry index, then do all pairs of x and everything ahead of it. Produces one less each pass, so there is no wasted compute.

[{'BDO', 'RBRTE'}, {'RWTC', 'RBRTE'}, {'RWTC', 'BDO'}]


In [55]:
df = pd.read_csv('../src/regime_detection/data/opec_meetings.csv', parse_dates= ['date'])
df_expanded = df.copy()
meeting_dates = df.copy()['date']
df_expanded = df.set_index('date').resample('D').ffill().reset_index()
df['opec_meeting'] = 1
df_expanded = df_expanded.merge(right=df, how='left', on ='date').fillna(0)

df_expanded['days_since_meeting'] = df_expanded["date"].apply(
        lambda d: (d - meeting_dates[meeting_dates <= d].max()).days
        if len(meeting_dates[meeting_dates <= d]) > 0 else None
    )

df_expanded

,date,opec_meeting,days_since_meeting
0,2007-03-15,1.0,0
1,2007-03-16,0.0,1
2,2007-03-17,0.0,2
3,2007-03-18,0.0,3
4,2007-03-19,0.0,4
...,...,...,...
7020,2026-06-03,0.0,185
7021,2026-06-04,0.0,186
7022,2026-06-05,0.0,187
7023,2026-06-06,0.0,188


In [56]:
def feature_pipeline(df=get_merged_df()):
    df = df.copy()
    df = compute_returns(df)
    df = series_diff(df)
    df = lagged_features(df)
    df = rolling_vol(df)
    df = differentials(df)
    df = rolling_z_scores(df)
    return df

feature_pipeline().columns

Index(['RBRTE', 'RWTC', 'T10Y2Y', 'VIXCLS', 'DTWEXBGS', 'WCESTUS1', 'RNGWHHD',
       'days_since_opec', 'RBRTE_log_return', 'RWTC_log_return',
       'RNGWHHD_log_return', 'WCESTUS1_w_change', 'RBRTE1_w_lag',
       'RBRTE4_w_lag', 'RBRTE12_w_lag', 'RWTC1_w_lag', 'RWTC4_w_lag',
       'RWTC12_w_lag', 'RBRTE_4_rol_vol', 'RBRTE_12_rol_vol', 'RWTC_4_rol_vol',
       'RWTC_12_rol_vol', 'RNGWHHD_4_rol_vol', 'RNGWHHD_12_rol_vol',
       'RBRTE_RWTC_diff', 'RBRTE_log_return_rol_z_score',
       'RWTC_log_return_rol_z_score', 'RNGWHHD_log_return_rol_z_score'],
      dtype='str')

In [57]:
import sys
print(sys.executable)

/Users/beniciouhart/regime_detection/.venv/bin/python


## ingestion.py

In [58]:
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from fredapi import Fred

import cloudscraper
import re
from datetime import datetime

load_dotenv()

def fetch_eia_series_pet(series_ids=['RBRTE', 'RWTC'], # ID's of series we're pulling (Brent, WTI)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)


    URL_BASE = f"https://api.eia.gov/v2/petroleum/pri/spt/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False) # Extract data from JSON output
    df['period'] = pd.to_datetime(df['period']) # Clean the period (date) to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']] # Keep only relevant columns

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table, rows indexed by date, columns are each ticker, and values are spot prices
    df_wide = df_wide.dropna()
    return df_wide

def fetch_fred_series(series_ids=['T10Y2Y','VIXCLS','DTWEXBGS'], #ID's of the series we want to pull, here 10Y-2Y spread, VIX, and DXY
                      frequency = 'W-FRI'  # How frequent we want the samples to be. (day -> "D", weekly (Friday) -> "W-FRI", monthly -> "M", yearly -> "Y"
                      ):
    
    load_dotenv()
    FRED_API_KEY = os.getenv('FRED_API_KEY')
    fred = Fred(api_key= FRED_API_KEY)
    series_dict = {}
    for id in series_ids: # Create a dict with key as ID and values as the corresponding series
        series_dict[id] = fred.get_series(series_id=id)

    df = pd.DataFrame(series_dict) # Turn dictionary into a DF
    df.index = pd.to_datetime(df.index) # Convert index dtype to datetime
    df.index.name = 'period' # Convert index name to 'period' to match the EIA information

    df = df.resample(rule=frequency).last() # Resample to keep only the days at the end of the week

    df = df.dropna() # Drop any NaNs

    return df

def fetch_eia_stock(series_ids=['WCESTUS1'], # ID's of series we're pulling (Week-end US Crude Inventory)
                          frequency= 'weekly',# Frequency of the spot prices, either daily, weekly, or monthly.
                            length = 5000 # Timespan that we're pulling, max 5000 weeks
                            ):
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)

    URL_BASE = f'https://api.eia.gov/v2/petroleum/stoc/wstk/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}'

    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide

def fetch_eia_series_gas(series_ids = ["RNGWHHD"], # ID's of series we're pulling (Henry Hub)
                         frequency= 'weekly', # Frequency of the spot prices, either daily, weekly, or monthly.
                         length = 5000): # Timespan that we're pulling, max 5000 weeks
    load_dotenv()
    EIA_API_KEY = os.getenv("EIA_API_KEY") #API Key to pull this data

    def series_line(series_ids) -> str: # Build the text used to specify what series we want
        text = [f"&facets[series][]={id}" for id in series_ids]
        return ''.join(text)
    

    URL_BASE = f"https://api.eia.gov/v2/natural-gas/pri/fut/data/?api_key={EIA_API_KEY}&frequency={frequency}&data[0]=value{series_line(series_ids)}&sort[0][column]=period&sort[0][direction]=desc&length={length}"
    response = requests.get(url= URL_BASE) # Make request to EIA API
    data = response.json()

    df = pd.DataFrame(data['response']['data']).sort_index(ascending= False)
    df['period'] = pd.to_datetime(df['period']) # Clean the period to datetime type
    df['value'] = pd.to_numeric(df['value'], errors= 'coerce') # Clean the value (spot price) to numeric
    df = df[['period', 'value', 'series']]

    df_wide = df.pivot(index='period', columns = 'series', values= 'value') # Create into pivot table
    df_wide = df_wide.dropna()
    return df_wide



def get_merged_df(eia_spot_pet_df=None, eia_spot_gas_df=None, eia_stock_df=None, fred_df=None):
    #Get data
    if eia_spot_pet_df is None:
        eia_spot_pet_df = fetch_eia_series_pet()
    if eia_spot_gas_df is None:
        eia_spot_gas_df = fetch_eia_series_gas()
    if eia_stock_df is None:
        eia_stock_df = fetch_eia_stock()
    if fred_df is None:
        fred_df = fetch_fred_series()

    # Merge data




    df = pd.merge(left=eia_spot_pet_df, right=fred_df,left_index = True, right_index = True, how= 'inner')
    df = pd.merge(left=df, right=eia_stock_df, left_index = True, right_index = True, how = 'inner')
    df = pd.merge(left=df, right= eia_spot_gas_df, left_index = True, right_index = True, how = 'inner' )
    df = df.copy().reset_index().sort_values('period')

    # Get and merge OPEC, FOMC meeting dates, and the time since a meeting (these are big events for the market)
        
    opec_dates = pd.read_csv('../src/regime_detection/data/opec_meetings.csv', parse_dates=['date']).sort_values('date')
    fomc_dates = pd.read_csv('../src/regime_detection/data/fomc_meetings.csv', parse_dates=['date']).sort_values('date')
    
    df = pd.merge_asof(df, opec_dates.assign(last_opec=opec_dates['date']),
                    left_on='period', right_on='date', direction='backward') #merge_asof used for timeseries data where the dates don't match, finds the closest date in the rightmost df and returns that value for all rows matches, here we use backward to get the most recent date, forward would get the soonest.
    df = pd.merge_asof(df, fomc_dates.assign(last_fomc=fomc_dates['date']), left_on='period', right_on='date', direction='backward')
    df['days_since_opec'] = (df['period'] - df['last_opec']).dt.days

    df['days_since_fomc'] = (df['period'] - df['last_fomc']).dt.days
    df = df.drop(columns=['date_x','date_y', 'last_fomc','last_opec'])
    
    return df.set_index('period')

get_merged_df().head()

,RBRTE,RWTC,T10Y2Y,VIXCLS,DTWEXBGS,WCESTUS1,RNGWHHD,days_since_opec,days_since_fomc
period,,,,,,,,,
2006-01-06,61.72,63.39,0.02,11.00,100.0241,302584,9.42,NaN,24
2006-01-13,62.18,63.74,0.02,11.23,99.9675,305325,8.63,NaN,31
2006-01-20,63.54,66.79,0.00,14.56,99.9017,303016,8.67,NaN,38
2006-01-27,63.77,66.82,0.01,11.97,99.6433,304935,8.22,NaN,45
2006-02-03,64.00,66.59,-0.05,12.96,100.1180,304523,8.36,NaN,3


In [59]:
import cloudscraper
import re
from datetime import datetime
from  bs4 import BeautifulSoup
import requests


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

response = requests.get("https://www.forexfactory.com/calendar/437-opec-meetings", headers=headers)
print(response.status_code)
print(response.text[:500])

soup = BeautifulSoup(response.text, 'html.parser') # Extract the text from the website

soup



403
e��:44(�|��?"��<�-�.�:�X�3e��߿�r\6�+.�Ͷ�]��n�mw��w���ﺺ8���� �4n�-�e���I״��j�{"q'�mMg�FZv�b	�*�K�0>��T���ywe�.fK�ܔ��Z�e[.f2�z��@t��ق�c�-fz{]�~��)T���(ʩ|Z���t��,I�i�����2��ϴ%$�$5,Wm�<Ѧ�U�Y6������eq�"�O >x�\��p�A��ZO~_Y�pԑ0�H]р .a��TQ�V���=��Ԡ^Vv~���^��]���y2q�S�+A�硸���,�[ :Y'#(���L������f��{�qT"


� d����~YopzGi���{���������
B�S��{�����W`,2���-*-��"R��4�v��5���s��d��//��H��=�2F@���6�6���c��͘}����6n3�y���J(�z��&amp;�c��ͳ-�E�n`������Pl�
S�&gt;$?�{]�
~��)T���(ʩ
|Z���t��,I�i�
����2��ϴ%$
�$5,Wm�&lt;Ѧ�U�Y6������eq�"�O &gt;x�
\��p�A��ZO~_Y�pԑ0�H]р .a��TQ�V���=��Ԡ^Vv~���^��
]���y2q�S��+A�硸���,�[ :Y'#(���L������f��{�qT"
e��:�44(�|��?"��&lt;�-�.�:�X�3e��߿�r\6�+.�Ͷ
�]��n�mw��w���ﺺ8���� �4n�-
�e���I״
��j�{"q'�mMg�FZv�b	�*�K�0
&gt;��T���ywe�.f
K�ܔ��Z�e[.f2�z��
@t��ق�c�-fzٖ��f֦?�b�UU�����2�gt�э)w�lF���l-~�
�28n�
��n�:�0M�p
�m
�K�o4q�lG��S;%[�
][����
�G���
�mA?�a�����"?������~�wٖՆ�a���G	���u]�9�M&lt;�~� ����~�|Ђ
l߿7}�s̳�
�P�����S�=v�
��|�&lt;��U�'��&amp;����뵄4a��s/Vd�r�ky6��+!�[�CD��P^��:yָ"���s
��Y�#�mK��I��6)�#b�iJ�"�fj��(7*"��rM��L��D	WD�&amp;ܨ܀�
���{�&amp;j�5�L)� �o�;M�9T�E
?X���Xo�À���
���$�D�	�Rb&gt;=]3�N5J%JM�~
G]�(
F]�p=�A�@T:��`���A
'�j�i�?�ɝ�}7
j�M��^!�&amp;�A���IL;]�H5Ii]�E��yoq�'�Ҫ�F�G7

## FOMC Date Scraper

In [60]:
import cloudscraper
import re
from datetime import datetime

def get_fomc_dates():

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
    }

    scraper = cloudscraper.create_scraper(
        interpreter='nodejs',  # Use Node.js instead of native solvers for tougher challenges
        browser={
            'browser': 'chrome',
            'platform': 'linux',
            'desktop': True
        })

    html = scraper.get(url='https://en.wikipedia.org/wiki/History_of_Federal_Open_Market_Committee_actions', headers=headers) # Scrape the wikipedia page for Historical FOMC Actions
    html_content = html.text # Extract the text from the website


    '''
    Match on format:
    (<td>)(Capital Letter)(One or more lowercase letters)(Space)(One or two digits)(Comma)(Space)(Four Digits)(Zero or more spaces)(</td>)
    '''
    pattern = r"<td>[A-Z][a-z]+\s\d{1,2},\s\d{4}\s*\n</td>"
    dates = re.findall(pattern, html_content)

    '''
    Strip the <td> / </td> labels and spaces in the text, I pattern match using them as
    I want to make sure to only pull the dates that were in the table.

    Then we convert the text to datetime format with strptime,
    and turn it back to our desired format with strftime.
    '''

    dates = [datetime.strptime(date.strip("<td>/ "),"%B %d, %Y ").strftime("%Y-%m-%d")for date in dates]

    dates_df = pd.DataFrame({"date": sorted(set(dates))}) # Turn sorted unique dates into a df

    return dates_df

get_fomc_dates().to_csv("../src/regime_detection/data/fomc_meetings.csv", index= False)

## OPEC Date Scraper

In [62]:
import cloudscraper
import re
from datetime import datetime
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

scraper = cloudscraper.create_scraper(
    interpreter='nodejs',  # Use Node.js instead of native solvers for tougher challenges
    browser={
        'browser': 'chrome',
        'platform': 'linux',
        'desktop': True
    })

html = scraper.get(url='https://www.forexfactory.com/calendar/437-opec-meetings') # Scrape the wikipedia page for Historical FOMC Actions
soup = BeautifulSoup(html.text, "html.parser")
print(html.text)

history_table = soup.find_all('table', class_="calendar-event__history alternating calendarhistory")[0]
links = history_table.find_all('a')
for idx, link in enumerate(links):
    print(f"Index: {idx}, Link: {link}")








<!DOCTYPE html> <html class="no-js" lang="en"> <head> <base href="https://www.forexfactory.com"> <meta charset="windows-1252"> <meta property="og:title" content="OPEC Meetings" /> <meta property="og:description" content="OPEC nations represent around 31% of the world&#039;s oil supply and are unified in their oil production levels. With so much control over oil&#039;s supply-side, shifts in their production levels can have a significant impact on oil prices;" /> <meta property="og:type" content="website" /> <meta property="og:url" content="https://www.forexfactory.com/calendar/437-opec-meetings" /> <meta property="og:image" content="https://resources.faireconomy.media/images/logos/bookmark_ff_400.png" /> <meta property="og:site_name" content="Forex Factory" /> <meta name="keywords" content="opec meetings the organization petroleum exporting countries misc global worldwide gasoline distillate wti crude oil iran iranian iraq kuwait saudi arabia venezuela libya algeria nigeria gabon equat

## Feature Creation and Pipeline

In [63]:
def compute_returns(df, series_ids= ['RBRTE','RWTC', 'RNGWHHD']):
    df = df.copy()
    for series in series_ids:
        df[series + '_log_return'] = np.log(df[series]/ df[series].shift(1))
    return df


def series_diff(df, series_ids=['WCESTUS1']):
    df = df.copy()
    for series in series_ids:
        df[series + '_1_w_change'] = df[series].diff()
    return df

def lagged_features(df, series_ids= ['RBRTE_log_return','RWTC_log_return', 'RNGWHHD_log_return', 'WCESTUS1_1_w_change'], lags=[1,4,12]):
    df = df.copy()
    for series in series_ids:
        for lag in lags:
            df[series + f'_{lag}_w_lag'] = df[series].shift(lag)
    return df

def rolling_vol(df, series_ids=['RBRTE_log_return','RWTC_log_return', 'RNGWHHD_log_return'], periods=[4,12]):
    df = df.copy()
    for series in series_ids:
        for period in periods:
            df[series + f'_{period}_returns_rol_vol'] = df[series].rolling(window=period,min_periods=1).std()
    return df

def differentials(df, series_ids=['RBRTE', 'RWTC']):
    df = df.copy()
    pairings = [(x,y) for i, x in enumerate(series_ids) for y in series_ids[i+1:]] 
    for pair in pairings:
        series_a , series_b = pair 
        df[f'{series_a}_{series_b}_diff'] = df[series_a] - df[series_b]
    return df

def rolling_z_scores(df, 
                     series_ids=['RBRTE_log_return','RWTC_log_return','RNGWHHD_log_return','RBRTE_RWTC_diff'],
                     window=63,
                     min_periods=1):
    df = df.copy()
    for series in series_ids:
      avg = df[series].rolling(window=window, min_periods=min_periods).mean()
      dev = df[series].rolling(window=window, min_periods=min_periods).std()
      df[series + '_rol_z_score'] = (df[series] - avg) / dev
    return df

def feature_pipeline(df=None):
    if df is None:
        df = get_merged_df()
    df = df.copy()
    print("Computing log returns...")
    df = compute_returns(df)
    print("Computing week-on-week changes...")
    df = series_diff(df)
    print("Computing lagged values...")
    df = lagged_features(df)
    print("Computing rolling returns volatility...")
    df = rolling_vol(df)
    print("Computing commodity-to-commodity differences...")
    df = differentials(df)
    print("Computing rolling window z-scores...")
    df = rolling_z_scores(df)
    return df


## ADF and KPSS Tests

In [75]:
from statsmodels.tsa.stattools import adfuller, kpss
import statsmodels as sm
import warnings

def adf_test(series,regression):
    """Augmented Dickey-Fuller: H0 = Non-Stationary. 
                         Reject H0 -> Stationary"""
    series = series.dropna()

    stat, pvalue, used_lag, nobs, crit, _ = adfuller(series,autolag='AIC', 
                                                     regression=regression)
    return {
        'test' : 'adf',
        'statistic' : stat,
        'pvalue' : pvalue,
        'lags' : used_lag,
        'nobs' : nobs,
        'regression' : regression,
        'is_stationary' : pvalue < 0.05,
    }

def kpss_test(series, regression):
    """KPSS Test: H0: Stationary or Trend-Stationary (Value is a function of time). 
           Reject H0 -> Non-Stationary"""
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        stat, pvalue, used_lag, crit = kpss(series, regression=regression, nlags='auto')
        interp_warning = any('InterpolationWarning' in str(wi.category) for wi in w)
    return {
        'test': "kpss",
        'statistic' : stat,
        'pvalue' : pvalue,
        'lag' : used_lag,
        'regression' : regression,
        'is_stationary' : pvalue >= 0.05, # P-Val >= 5% so that we fail to reject the null
        'pvalue_clipped' : interp_warning,
    }

def classify_stationarity(series, regression='ct'):
    adf_res = adf_test(series, regression)
    kpss_res = kpss_test(series, regression)
    adf_stat, kpss_stat = adf_res['is_stationary'], kpss_res['is_stationary']

    if adf_stat and kpss_stat:
        verdict = 'stationary'
    elif not adf_stat and kpss_stat:
        verdict = 'trend_stationary'
    elif adf_stat and not kpss_stat:
        verdict = 'difference_stationary'
    else:
        verdict = 'non_stationary'
    return {
        'adf_pvalue' : adf_res['pvalue'],
        'adf_stationary' : adf_res['is_stationary'],
        'kpss_pvalue' : kpss_res['pvalue'],
        'kpss_stationary' : kpss_res['is_stationary'],
        'kpss_pvalue_clipped' : kpss_res['pvalue_clipped'],
        'verdict' : verdict,
    }

def transform_and_revalidate(series, verdict, regression='ct'):
    """ 
    Takes in a series and it's stattionarity verdict.

    It returns a series transformed with regards to the verdict,
    along with what transformation was applied.

    """
    if verdict == 'stationary':
        transformed, method = series, 'none'

    elif verdict == 'trend_stationary':
        # Create an array of the index (time) of each row in the series
        t = np.arange(len(series))
        """
        Fit an OLS model with series values as dependent variable 
        and the time as the independent var.

        Subtract the line from the actual series to 'detrend' it.
        
        """
        resid = series - sm.OLS(series, sm.add_constant(t)).fit().fitted_values
        transformed, method = resid, 'detrended'

    elif verdict in ('non_stationary', 'difference_stationary'):
        # Simply difference to remove unit root.
        transformed = series.diff().dropna()
        transformed, method =  transformed, 'differenced'

    post = classify_stationarity(transformed, regression)

    return transformed, method, post
    
series_ids= ['RBRTE_log_return', 'RWTC_log_return',
             'RNGWHHD_log_return','WCESTUS1_1_w_change', 
             'T10Y2Y', 'VIXCLS', 'DTWEXBGS']

df = feature_pipeline().dropna()


for series in series_ids:
    stationary = classify_stationarity(df[series])
    print(stationary)
    print(transform_and_revalidate(df[series], verdict= stationary['verdict']))
    
    

Computing log returns...
Computing week-on-week changes...
Computing lagged values...
Computing rolling returns volatility...
Computing commodity-to-commodity differences...
Computing rolling window z-scores...
{'adf_pvalue': np.float64(1.3163087767322404e-16), 'adf_stationary': np.True_, 'kpss_pvalue': np.float64(0.1), 'kpss_stationary': np.True_, 'kpss_pvalue_clipped': True, 'verdict': 'stationary'}
(period
2007-03-16    0.008579
2007-03-23    0.003608
2007-03-30    0.078821
2007-04-06    0.036395
2007-04-13   -0.005119
                ...   
2026-05-15    0.042981
2026-05-22    0.000724
2026-05-29   -0.130784
2026-06-05    0.019388
2026-06-12   -0.053876
Name: RBRTE_log_return, Length: 1005, dtype: float64, 'none', {'adf_pvalue': np.float64(1.3163087767322404e-16), 'adf_stationary': np.True_, 'kpss_pvalue': np.float64(0.1), 'kpss_stationary': np.True_, 'kpss_pvalue_clipped': True, 'verdict': 'stationary'})
{'adf_pvalue': np.float64(2.1018946715525464e-19), 'adf_stationary': np.True_

In [ ]:
random_seed = np.random.seed(42)
def kpss_test(df,series_ids = ['RBRTE_log_return', 'RWTC_log_return',
       'RNGWHHD_log_return','WCESTUS1_1_w_change', 'T10Y2Y', 'VIXCLS', 'DTWEXBGS']):
    results = {}
    for series in series_ids:
        test = kpss(df[series])
        results[series] = test
        print(f"KPSS Test Statistic: {test[0]}")
        print(f"P-Value: {test[1]}")
        print(f"Number of lags: {test[2]}")
        print(f"Critical Value: {test[3]}")

kpss_test(get_merged_df())

